# Country-Level Longer-Term Forecasting of First-Time Asylum Applications

*A time-series analysis and forecasting exercise using Italy monthly Eurostat data*

This notebook uses monthly first-time asylum application data for Italy to demonstrate a longer-term country-level forecasting workflow with real administrative time-series data.

The data source is [Eurostat's monthly asylum applications dataset](https://ec.europa.eu/eurostat/databrowser/view/migr_asyappctzm/default/table?lang=en), using the Italy series for first-time asylum applicants.

Eurostat defines a first-time asylum applicant as a third-country national or stateless person who submitted, or was included in, an application for international protection during the reference period and is applying for international protection for the first time. The unit is persons, the reference period is calendar month/year, and Eurostat rounds the application data to the nearest 5.

The purpose is demonstrative and pedagogical. The notebook is not an operational forecast product, and it does not attempt to explain the full political, legal, administrative, or structural dynamics behind asylum applications in Italy.

Asylum applications are administrative events. They are not the same as regular or irregular entries, border arrivals, total displacement movements, or the number of people in need of protection at any given time. They may be affected by access to the asylum procedure, registration capacity, policy rules, onward movement, and wider dynamics in Italy's asylum system or in countries of origin.

The forecasts should therefore be read as model-based projections of an administrative series, not as direct predictions of future arrivals or international protection needs.

# Part I - Understanding the time series

## 1. Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.seasonal import seasonal_decompose


## 2. Load prepared monthly series

The public notebook loads a cleaned monthly CSV. The source data were downloaded from Eurostat and prepared from a local Excel extract into a monthly Italy series.

The Excel extract had countries as rows and monthly values as columns, with monthly date columns alternating with status or blank columns. During preparation, the Italy row was selected, monthly columns were parsed, Eurostat missing markers were converted, and incomplete months were excluded.

The modelling series used here runs from **January 2008 to February 2026**. The incomplete months **March 2026** and **April 2026** are not imputed and are not included.


In [ ]:
data_path_candidates = [
    Path("data/processed/italy_first_time_asylum_monthly.csv"),
    Path("../../data/processed/italy_first_time_asylum_monthly.csv"),
]
data_path = next(path for path in data_path_candidates if path.exists())

applications = (
    pd.read_csv(data_path, parse_dates=["date"])
    .sort_values("date")
    .reset_index(drop=True)
)

applications_display = applications.rename(
    columns={"date": "Month", "applications": "First-time applications"}
)
applications_display.head()


In [ ]:
applications_display.tail()


Before plotting the series, we check the basic structure of the data.


In [ ]:
summary_table = pd.DataFrame(
    {
        "Measure": [
            "Start date",
            "End date",
            "Number of months",
            "Total applications",
            "Mean monthly applications",
            "Median monthly applications",
            "Minimum monthly applications",
            "Maximum monthly applications",
            "Missing values",
        ],
        "Value": [
            applications["date"].min().strftime("%B %Y"),
            applications["date"].max().strftime("%B %Y"),
            len(applications),
            f"{applications['applications'].sum():,.0f}",
            f"{applications['applications'].mean():,.1f}",
            f"{applications['applications'].median():,.1f}",
            f"{applications['applications'].min():,.0f}",
            f"{applications['applications'].max():,.0f}",
            int(applications.isna().sum().sum()),
        ],
    }
)

summary_table


The series contains monthly counts of first-time asylum applications in Italy. The values are non-negative integer counts. The date range and missing-value checks confirm that the processed file is ready for exploratory time-series analysis.

## 3. Exploratory time-series analysis

We start by plotting the raw monthly series.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications["date"],
    applications["applications"],
    linewidth=1.2,
)
ax.set_title("Italy monthly first-time asylum applications")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.25)
fig.tight_layout()


The raw series shows substantial variation over time. Some periods have relatively low monthly application counts, while others show much higher pressure. This is expected for an asylum application series: the observed counts can respond to conflict dynamics, route dynamics, access to territory and procedures, policy changes, registration practices, and administrative capacity.

The raw monthly pattern is useful, but it is also noisy. To make the longer movement easier to see, we add a 12-month rolling average.


In [ ]:
applications_with_rolling = applications.assign(
    rolling_12_month_average=applications["applications"].rolling(12).mean()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["applications"],
    linewidth=0.9,
    alpha=0.45,
    label="Monthly applications",
)
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["rolling_12_month_average"],
    linewidth=2.0,
    label="12-month rolling average",
)
ax.set_title("Italy first-time asylum applications with 12-month rolling average")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


The 12-month rolling average smooths short-term month-to-month variation and makes broader phases more visible. It does not remove the underlying uncertainty or explain the causes of change. It simply helps separate longer movements from monthly fluctuation.

For an annual view, we aggregate the monthly counts by calendar year.


In [ ]:
annual_applications = (
    applications.assign(year=applications["date"].dt.year)
    .groupby("year", as_index=False)["applications"]
    .sum()
    .rename(columns={"year": "Year", "applications": "Total applications"})
)

annual_applications


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    annual_applications["Year"].astype(str),
    annual_applications["Total applications"],
)
ax.set_title("Annual first-time asylum applications in Italy")
ax.set_xlabel("Year")
ax.set_ylabel("Applications")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()


The annual summary gives a clearer sense of scale. The series includes quieter years, higher-pressure years, and periods where the level changes substantially. A constant-average forecast is therefore unlikely to be sufficient.

The next step is to define an evaluation design before fitting forecasting models.


# Part II - Creating an evaluation framework


## 4. Train/test split

Forecasting models should be evaluated against observations that were not used during fitting. In a time-series setting, this means holding back later observations as a future test period.

This notebook uses a chronological train/test split:

- **Training period:** January 2008 to February 2023
- **Test period:** March 2023 to February 2026

The test period covers 36 months, or three full seasonal cycles. This gives a held-out period for comparing the baselines, SARIMA models, and SARIMAX model against observations that were not used during fitting.


In [ ]:
train_start = pd.Timestamp("2008-01-01")
train_end = pd.Timestamp("2023-02-01")
test_start = pd.Timestamp("2023-03-01")
test_end = pd.Timestamp("2026-02-01")

train = applications.loc[
    applications["date"].between(train_start, train_end)
].copy()
test = applications.loc[
    applications["date"].between(test_start, test_end)
].copy()

assert train["date"].min() == train_start
assert train["date"].max() == train_end
assert test["date"].min() == test_start
assert test["date"].max() == test_end
assert len(test) == 36

split_summary = pd.DataFrame(
    {
        "Period": ["Training", "Test"],
        "Start": [
            train["date"].min().strftime("%Y-%m"),
            test["date"].min().strftime("%Y-%m"),
        ],
        "End": [
            train["date"].max().strftime("%Y-%m"),
            test["date"].max().strftime("%Y-%m"),
        ],
        "Number of months": [len(train), len(test)],
        "Total applications": [
            int(train["applications"].sum()),
            int(test["applications"].sum()),
        ],
        "Mean monthly applications": [
            round(train["applications"].mean(), 1),
            round(test["applications"].mean(), 1),
        ],
    }
)

split_summary


The split is shown visually below. The vertical line marks the boundary between the training period and the held-out test period.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    train["date"],
    train["applications"],
    label="Training period",
    linewidth=1.2,
)
ax.plot(
    test["date"],
    test["applications"],
    label="Test period",
    linewidth=1.2,
)
ax.axvline(
    test_start,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Train/test boundary",
)
ax.set_title("Italy monthly first-time asylum applications: train/test split")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


## 5. Seasonal decomposition on the training data only

Before fitting forecasting models, it is useful to inspect the structure of the training series.

Seasonal decomposition separates a time series into three broad components:

- a trend component, which shows slower-moving changes in the level of the series;
- a seasonal component, which shows recurring within-year patterns;
- a residual or irregular component, which is what remains after the trend and seasonal pattern are removed.

The decomposition below is run on the training data only. This is important: the test period is supposed to represent future observations, so it should not be used to inspect or tune the model before evaluation.

The decomposition is descriptive. It is not yet a forecast model.


In [ ]:
training_series = train.set_index("date")["applications"].asfreq("MS")

decomposition = seasonal_decompose(
    training_series,
    model="additive",
    period=12,
)

decomposition_df = pd.DataFrame(
    {
        "observed": decomposition.observed,
        "trend": decomposition.trend,
        "seasonal": decomposition.seasonal,
        "residual": decomposition.resid,
    }
)

fig, axes = plt.subplots(4, 1, figsize=(10, 8), sharex=True)
fig.suptitle(
    "Seasonal decomposition of asylum applications within the training period\n"
    "Training period: 2008-01 to 2023-02",
    y=0.99,
)

components = [
    ("observed", "Observed training series", "#1f77b4"),
    ("trend", "Smoothed trend component", "#2ca02c"),
    ("seasonal", "Estimated seasonal component", "#ff7f0e"),
    ("residual", "Residual / irregular component", "#d62728"),
]

for ax, (column, title, color) in zip(axes, components):
    ax.plot(
        decomposition_df.index,
        decomposition_df[column],
        color=color,
        linewidth=1.2,
    )
    ax.set_title(title, loc="left")
    ax.grid(alpha=0.25)

axes[2].axhline(0, color="black", linewidth=0.8, alpha=0.7)
axes[3].axhline(0, color="black", linewidth=0.8, alpha=0.7)
axes[-1].set_xlabel("Month")
fig.tight_layout()
plt.show()


The decomposition gives three useful signals.

First, the trend component changes substantially over time. It is relatively low in the early years, rises through the mid-2010s, reaches its highest level around 2016-2017, then declines before rising again toward the end of the training period.

Second, a recurring seasonal pattern is visible. Some months tend to sit above or below the average within a year. The decomposition does not identify the cause, but it does show that within-year variation is present in the training data.

Third, the residual component still contains large irregular movements. Trend and seasonality therefore do not explain the full series. The forecasting task has to deal with structural changes and irregular shocks as well as recurring seasonal structure.

The next step is to compare simple baseline forecasts before fitting SARIMA or SARIMAX models.


## 6. Naive baseline forecasts

Before fitting SARIMA or SARIMAX models, we create three simple fixed-origin benchmark forecasts.

These baselines set a minimum standard for later models. A more complex model should add value compared with transparent alternatives that are easy to understand.

The three baselines are:

- **last-value baseline:** every test month is forecast as the final observed value in the training period;
- **12-month moving-average baseline:** every test month is forecast as the average of the final 12 months of the training period;
- **seasonal naive baseline:** the final 12 observed training months are repeated across the test period.

Fixed-origin means that the forecasts are made using only information available at the end of the training period, February 2023, and then extended across the held-out test period. This is a strict, leakage-free benchmark setup. It is not meant to represent how a real operation would forecast for three years without updating.

We evaluate the baselines using two metrics:

- **MAE**, or mean absolute error, which gives the average monthly forecast error in applications;
- **RMSE**, or root mean squared error, which gives more weight to larger forecast misses.

MAE is the primary metric because it is easy to interpret in the same unit as the series: monthly first-time asylum applications. RMSE is shown as a secondary check.

The notebook uses a few small project helper functions for repeated forecast evaluation and plotting. These keep the modelling sections focused on the analytical choices rather than repeated display boilerplate.


In [ ]:
import sys

src_path = next(
    candidate / "src"
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "src" / "evaluation" / "forecast_evaluation.py").exists()
)
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from evaluation.forecast_evaluation import (  # noqa: E402
    display_metric_table,
    evaluate_forecast,
    make_forecast_frame,
    make_model_info_table,
    plot_forecast_with_interval,
)


We now create the three baseline forecasts for the held-out test period.


In [ ]:
last_training_value = train["applications"].iloc[-1]
last_12_month_average = train["applications"].tail(12).mean()
seasonal_pattern = train["applications"].tail(12).to_list()
seasonal_repetitions = (len(test) // len(seasonal_pattern)) + 1
seasonal_naive_forecast = (seasonal_pattern * seasonal_repetitions)[: len(test)]

baseline_forecasts = pd.DataFrame(
    {
        "date": test["date"].to_numpy(),
        "actual_applications": test["applications"].to_numpy(),
        "last_value_forecast": last_training_value,
        "moving_average_12_month_forecast": last_12_month_average,
        "seasonal_naive_forecast": seasonal_naive_forecast,
    }
)


The plot below compares the actual held-out test observations with the three baseline forecasts.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["actual_applications"],
    label="Actual test observations",
    linewidth=1.8,
)
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["last_value_forecast"],
    label="Last-value baseline",
    linewidth=1.2,
)
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["moving_average_12_month_forecast"],
    label="12-month moving-average baseline",
    linewidth=1.2,
)
ax.plot(
    baseline_forecasts["date"],
    baseline_forecasts["seasonal_naive_forecast"],
    label="Seasonal naive baseline",
    linewidth=1.2,
)
ax.set_title("Naive baseline forecasts over the held-out test period")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


The error table summarises baseline performance over the full held-out test period.


In [ ]:
baseline_metrics = pd.DataFrame(
    [
        evaluate_forecast(
            baseline_forecasts["actual_applications"],
            baseline_forecasts["last_value_forecast"],
            "Last-value",
            label_column="Baseline",
        ),
        evaluate_forecast(
            baseline_forecasts["actual_applications"],
            baseline_forecasts["moving_average_12_month_forecast"],
            "12-month moving average",
            label_column="Baseline",
        ),
        evaluate_forecast(
            baseline_forecasts["actual_applications"],
            baseline_forecasts["seasonal_naive_forecast"],
            "Seasonal naive",
            label_column="Baseline",
        ),
    ]
)

display_metric_table(baseline_metrics)


The last-value baseline performs best over the held-out test period, with the lowest MAE and RMSE among the three simple benchmarks. In this test period, carrying forward the February 2023 level performs better than using the previous 12-month average or repeating the final observed seasonal pattern.

The 12-month moving-average and seasonal naive baselines both sit too low for much of the test period. The seasonal naive forecast repeats a within-year pattern, but because it repeats a lower training-period level, it misses the higher level seen in much of the test period.

This gives the first SARIMA model a clear reference point: it needs to improve on the last-value baseline, not only on the weaker moving-average or seasonal naive baselines.


# Part III - Model-based forecasting and evaluation

## 7. First SARIMA model

We now move from benchmark forecasts to a first SARIMA model.

SARIMA stands for seasonal autoregressive integrated moving-average model. In practical terms, it uses past values, past forecast errors, differencing, and seasonal structure to model a time series. This makes it a useful first modelling step for a monthly series with changing levels and recurring within-year patterns.

This first model is deliberately simple. It is not the final model selection. It gives us an initial model-based benchmark before we compare a small set of alternative SARIMA specifications.

The first SARIMA structure is:

```text
SARIMA(1, 1, 1)(1, 1, 1, 12)
```

The non-seasonal part, `(1, 1, 1)`, allows for one autoregressive term, one order of differencing, and one moving-average term. The seasonal part, `(1, 1, 1, 12)`, applies the same basic idea at a 12-month seasonal cycle.


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX


In [ ]:
training_series = train.set_index("date")["applications"].asfreq("MS")


We fit the first SARIMA model below.


In [ ]:
first_sarima_model = SARIMAX(
    training_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

first_sarima_results = first_sarima_model.fit(disp=False)


The fitted model is summarised in a compact table.


In [ ]:
first_sarima_model_info = make_model_info_table(
    "First SARIMA",
    (1, 1, 1),
    (1, 1, 1, 12),
    first_sarima_results,
    train,
    test,
)

first_sarima_model_info


We now forecast the 36-month test period. The forecast is generated from the model fitted only on the training data.

The plot shows the observed applications in the test period, the SARIMA point forecast, and an 80% model-based confidence interval.


In [ ]:
first_sarima_forecast = make_forecast_frame(
    first_sarima_results,
    test,
    forecast_column="sarima_forecast",
    lower_column="sarima_lower_80",
    upper_column="sarima_upper_80",
)


In [ ]:
plot_forecast_with_interval(
    first_sarima_forecast,
    forecast_column="sarima_forecast",
    lower_column="sarima_lower_80",
    upper_column="sarima_upper_80",
    title="First SARIMA forecast over the held-out test period",
    forecast_label="First SARIMA forecast",
    grid_alpha=0.3,
)


The shaded band is an 80% model-based interval. It reflects uncertainty under this fitted SARIMA specification; it does not include unmodelled changes in policy, access to procedure, administrative capacity, route dynamics, or geopolitical shocks. Because the model is continuous, lower bounds can sometimes approach or fall below zero; this should not be read as a literal negative-count forecast.


In [ ]:
first_sarima_metric = evaluate_forecast(
    first_sarima_forecast["actual_applications"],
    first_sarima_forecast["sarima_forecast"],
    "First SARIMA",
)

model_comparison_metrics = pd.concat(
    [
        baseline_metrics.rename(columns={"Baseline": "Baseline / model"}),
        pd.DataFrame([first_sarima_metric]),
    ],
    ignore_index=True,
)

display_metric_table(model_comparison_metrics)


The first SARIMA model improves on all three naive baselines over the held-out test period. Its MAE and RMSE are both lower than the last-value baseline, which was the strongest naive benchmark.

The forecast follows the broad level of the test period better than the baselines, but it remains smoother than the observed monthly series. It does not capture sharper month-to-month movements, including the pronounced dip in 2025.

The 80% interval widens over the forecast horizon, as expected when the model projects further beyond the training period. This first SARIMA model is therefore a useful starting point, but not yet the final specification.


## 8. SARIMA model selection

The first SARIMA model improved on the naive baselines, but it used only one manually chosen specification. SARIMA models require choices about the non-seasonal and seasonal components of the model. Rather than relying on one starting specification, we now compare a small set of plausible alternatives.

This is not an exhaustive search. The aim is to keep the search space limited, transparent, and computationally manageable. We vary only low-order SARIMA specifications and keep the seasonal period fixed at 12 months, because the data are monthly.

The grid search is run on the training period only. The held-out test period is still kept separate and will be used in the next section to evaluate the selected model.

The search space is:


In [ ]:
p_values = [0, 1, 2]
d_values = [0, 1]
q_values = [0, 1, 2]
P_values = [0, 1]
D_values = [0, 1]
Q_values = [0, 1]
seasonal_period = 12

total_sarima_candidates = (
    len(p_values)
    * len(d_values)
    * len(q_values)
    * len(P_values)
    * len(D_values)
    * len(Q_values)
)
assert total_sarima_candidates == 144

sarima_grid_definition = pd.DataFrame(
    [
        {"Parameter": "p", "Values": "0, 1, 2", "Meaning": "Non-seasonal autoregressive order"},
        {"Parameter": "d", "Values": "0, 1", "Meaning": "Non-seasonal differencing order"},
        {"Parameter": "q", "Values": "0, 1, 2", "Meaning": "Non-seasonal moving-average order"},
        {"Parameter": "P", "Values": "0, 1", "Meaning": "Seasonal autoregressive order"},
        {"Parameter": "D", "Values": "0, 1", "Meaning": "Seasonal differencing order"},
        {"Parameter": "Q", "Values": "0, 1", "Meaning": "Seasonal moving-average order"},
        {"Parameter": "s", "Values": "12", "Meaning": "Seasonal period in months"},
    ]
)

sarima_grid_definition


This gives 144 candidate SARIMA specifications:

```text
3 p-values x 2 d-values x 3 q-values x 2 P-values x 2 D-values x 2 Q-values = 144 candidates
```

Each candidate is fit to the training series and ranked by AIC. AIC is a model-selection criterion that balances fit and complexity: lower values are preferred, but AIC is still based on training-data fit. It does not by itself prove that a model will forecast the held-out test period better.


In [ ]:
import itertools
import time
import warnings

training_series = train.set_index("date")["applications"].asfreq("MS")

sarima_grid_rows = []
grid_search_start = time.perf_counter()

for p, d, q, P, D, Q in itertools.product(
    p_values, d_values, q_values, P_values, D_values, Q_values
):
    order = (p, d, q)
    seasonal_order = (P, D, Q, seasonal_period)

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            candidate_model = SARIMAX(
                training_series,
                order=order,
                seasonal_order=seasonal_order,
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            candidate_results = candidate_model.fit(disp=False)

        mle_retvals = getattr(candidate_results, "mle_retvals", {}) or {}
        sarima_grid_rows.append(
            {
                "order": order,
                "seasonal_order": seasonal_order,
                "AIC": candidate_results.aic,
                "BIC": candidate_results.bic,
                "converged": bool(mle_retvals.get("converged", False)),
                "failure_reason": None,
            }
        )
    except Exception as exc:
        sarima_grid_rows.append(
            {
                "order": order,
                "seasonal_order": seasonal_order,
                "AIC": pd.NA,
                "BIC": pd.NA,
                "converged": False,
                "failure_reason": str(exc),
            }
        )

sarima_grid_runtime_seconds = time.perf_counter() - grid_search_start
sarima_grid_results = pd.DataFrame(sarima_grid_rows)

fitted_sarima_grid_results = sarima_grid_results[
    sarima_grid_results["failure_reason"].isna()
].copy()

best_sarima_aic = fitted_sarima_grid_results["AIC"].min()
fitted_sarima_grid_results["Delta AIC"] = (
    fitted_sarima_grid_results["AIC"] - best_sarima_aic
)

sarima_top5 = (
    fitted_sarima_grid_results.sort_values(["AIC", "BIC"])
    .head(5)
    .reset_index(drop=True)
)
sarima_top5.insert(0, "rank", sarima_top5.index + 1)

selected_sarima_specification = sarima_top5.head(1).copy()
selected_sarima_order = selected_sarima_specification.loc[0, "order"]
selected_sarima_seasonal_order = selected_sarima_specification.loc[0, "seasonal_order"]


The grid-search run is summarised below.


In [ ]:
successful_sarima_fits = int(sarima_grid_results["failure_reason"].isna().sum())
failed_sarima_fits = int(sarima_grid_results["failure_reason"].notna().sum())

sarima_grid_summary = pd.DataFrame(
    [
        {
            "Total candidates": total_sarima_candidates,
            "Successful fits": successful_sarima_fits,
            "Failed fits": failed_sarima_fits,
            "Runtime seconds": round(sarima_grid_runtime_seconds, 1),
        }
    ]
)

sarima_grid_summary


The tables report four model-selection fields. **AIC** and **BIC** are information criteria: both reward better fit and penalise unnecessary complexity, with lower values preferred. BIC penalises complexity more strongly than AIC. **Delta AIC** shows how far each candidate is from the lowest-AIC model in this grid; values close to zero mean that the candidates are very similar by AIC. **Converged** indicates whether the numerical fitting routine reported successful convergence.

We show only the five best fitted specifications by AIC.


In [ ]:
sarima_top5_display = sarima_top5[
    ["rank", "order", "seasonal_order", "AIC", "Delta AIC", "BIC", "converged"]
].copy()

for column in ["AIC", "Delta AIC", "BIC"]:
    sarima_top5_display[column] = sarima_top5_display[column].round(1)

sarima_top5_display


The lowest-AIC specification in this limited grid search is SARIMA(0, 1, 2)(1, 1, 1, 12). It keeps first differencing and seasonal differencing, uses two non-seasonal moving-average terms, and includes seasonal autoregressive and moving-average terms at the 12-month cycle.

The top AIC results are close. The second-ranked model has a Delta AIC of only 0.2, and the third and fourth candidates are also within about 2 AIC points of the selected model. The selected specification is therefore the best candidate in this limited search, not the only reasonable choice.

All 144 candidates returned fitted results, with no failed fits. Six fitted models did not converge, but the selected model did. The next section evaluates whether the selected specification improves held-out forecast performance compared with the baselines and the first SARIMA model.


## 9. Selected SARIMA diagnostics and test-set evaluation

The previous section selected SARIMA(0, 1, 2)(1, 1, 1, 12) as the lowest-AIC specification within the limited grid search.

We now use that selected specification in two steps.

First, we briefly inspect the fitted model diagnostics on the training period. This is a statistical check of the model residuals, not the main evaluation criterion.

Second, we forecast the held-out test period and compare the selected SARIMA model against the naive baselines and the first SARIMA model.


### 9.1 Fit the selected SARIMA model

The selected SARIMA specification is fitted on the training series only.


In [ ]:
selected_sarima_model = SARIMAX(
    training_series,
    order=selected_sarima_order,
    seasonal_order=selected_sarima_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)
selected_sarima_results = selected_sarima_model.fit(disp=False)


The fitted selected model is summarised below.


In [ ]:
selected_sarima_model_info = make_model_info_table(
    "Selected SARIMA",
    selected_sarima_order,
    selected_sarima_seasonal_order,
    selected_sarima_results,
    train,
    test,
)

selected_sarima_model_info


### 9.2 Brief residual diagnostics

Before evaluating the selected model on the held-out test period, we briefly inspect its residual diagnostics.

The diagnostic plot gives a compact view of residual behaviour, including residuals over time, residual distribution, a Q-Q plot, and residual autocorrelation. This helps identify obvious problems, such as remaining time dependence or strong departures from the model assumptions.


In [ ]:
selected_sarima_results.plot_diagnostics(figsize=(10, 6))
plt.tight_layout()
plt.show()


The diagnostic plots do not show obvious remaining autocorrelation. The residual correlogram is mostly within the confidence band after lag zero, suggesting that the selected SARIMA model has captured much of the remaining time dependence in the training series.

The residuals are mostly centred around zero, but there are several large spikes. The histogram is roughly centred, while the Q-Q plot follows the reference line through the middle but departs in the tails. This suggests that the residuals are not perfectly normal and that some extreme months are not fully captured by the model.

These diagnostics are useful checks, but they are not the main evaluation criterion. The more important question for this notebook is whether the selected SARIMA improves forecast performance on the held-out test period.


### 9.3 Forecast the held-out test period

We now forecast the full held-out test period: March 2023 to February 2026.

The forecast is generated from the selected SARIMA model fitted only on the training data.


In [ ]:
selected_sarima_forecast = make_forecast_frame(
    selected_sarima_results,
    test,
    forecast_column="selected_sarima_forecast",
    lower_column="selected_sarima_lower_80",
    upper_column="selected_sarima_upper_80",
)


The plot compares the selected SARIMA forecast with the actual held-out test observations.


In [ ]:
plot_forecast_with_interval(
    selected_sarima_forecast,
    forecast_column="selected_sarima_forecast",
    lower_column="selected_sarima_lower_80",
    upper_column="selected_sarima_upper_80",
    title="Selected SARIMA forecast over the held-out test period",
    forecast_label="Selected SARIMA forecast",
)


The interval should be read in the same model-based sense as above.


### 9.4 Compare forecast accuracy

The final step is to compare the selected SARIMA model against the earlier benchmarks.

We use the same evaluation metrics as before:

- **MAE**, the average absolute forecast error in monthly applications;
- **RMSE**, which gives more weight to larger forecast misses.


In [ ]:
selected_sarima_metric = evaluate_forecast(
    selected_sarima_forecast["actual_applications"],
    selected_sarima_forecast["selected_sarima_forecast"],
    "Selected SARIMA",
)


In [ ]:
selected_model_comparison_metrics = pd.concat(
    [
        model_comparison_metrics,
        pd.DataFrame([selected_sarima_metric]),
    ],
    ignore_index=True,
)

display_metric_table(selected_model_comparison_metrics)


The selected SARIMA model performs slightly better than the first SARIMA model on the held-out test period. Its MAE falls from **1544.3** to **1526.4**, and its RMSE falls from **2029.3** to **2020.6**. The improvement is modest.

The selected SARIMA also performs clearly better than the strongest naive baseline, the last-value forecast. Compared with that baseline, it reduces MAE by about **498 applications** and RMSE by about **549 applications**.

The forecast follows the broad level of the test period reasonably well, but it remains smoother than the observed series and does not fully capture sharp monthly movements, especially the pronounced dip in 2025.

The selected SARIMA is therefore the strongest model so far. The next section tests whether adding descriptive event-period indicators through SARIMAX improves performance further.


## 10. SARIMAX with descriptive event-period indicators

The selected SARIMA model uses the past behaviour of the application series, including seasonal structure, but it does not include any additional information about historically unusual periods.

We now test a limited SARIMAX extension using three descriptive event-period indicators. These indicators mark broad historical periods that may have affected the level, timing, or administrative recording of first-time asylum applications in Italy.

They are not causal estimates. Their purpose is narrower: to test whether marking a few historically unusual periods helps the model estimate the training-period structure and improve held-out forecast performance.


We use three descriptive event-period indicators:

**`european_refugee_crisis` - 2015-01 to 2016-12**  
This indicator marks the 2015-2016 European refugee crisis, when asylum applications across Europe reached exceptional levels. It is included as a broad regional marker, not as an Italy-specific causal explanation.

**`central_med_route_shift` - 2017-07 to 2019-12**  
This indicator marks a period of changed Central Mediterranean route dynamics after mid-2017, including increased cooperation between Italian and Libyan authorities, more interceptions, reduced successful crossings to Italy, and a changed rescue/disembarkation environment. It is not interpreted as the causal effect of one agreement or policy, such as the Italy-Libya Memorandum of Understanding.

**`covid_disruption` - 2020-03 to 2021-12**  
This indicator marks the COVID-19 disruption period, when mobility restrictions, border measures, and administrative interruptions plausibly affected access to the asylum procedure and, consequently, the observed number of asylum applications.


The indicator columns are added to a copy of the main application table. The original `applications` dataframe is left unchanged.


In [ ]:
applications_with_indicators = applications.copy()

european_refugee_crisis_mask = (
    (applications_with_indicators["date"] >= pd.Timestamp("2015-01-01"))
    & (applications_with_indicators["date"] <= pd.Timestamp("2016-12-01"))
)
applications_with_indicators["european_refugee_crisis"] = (
    european_refugee_crisis_mask.astype(int)
)

central_med_route_shift_mask = (
    (applications_with_indicators["date"] >= pd.Timestamp("2017-07-01"))
    & (applications_with_indicators["date"] <= pd.Timestamp("2019-12-01"))
)
applications_with_indicators["central_med_route_shift"] = (
    central_med_route_shift_mask.astype(int)
)

covid_disruption_mask = (
    (applications_with_indicators["date"] >= pd.Timestamp("2020-03-01"))
    & (applications_with_indicators["date"] <= pd.Timestamp("2021-12-01"))
)
applications_with_indicators["covid_disruption"] = covid_disruption_mask.astype(int)



In [ ]:
assert int(applications_with_indicators["european_refugee_crisis"].sum()) == 24
assert int(applications_with_indicators["central_med_route_shift"].sum()) == 30
assert int(applications_with_indicators["covid_disruption"].sum()) == 22


The plot below shows where the three descriptive event periods fall in the monthly application series.


In [ ]:
indicator_plot_periods = [
    {
        "label": "European refugee crisis",
        "start": pd.Timestamp("2015-01-01"),
        "end": pd.Timestamp("2016-12-01"),
        "color": "tab:orange",
    },
    {
        "label": "Central Med route shift",
        "start": pd.Timestamp("2017-07-01"),
        "end": pd.Timestamp("2019-12-01"),
        "color": "tab:green",
    },
    {
        "label": "COVID-19 disruption",
        "start": pd.Timestamp("2020-03-01"),
        "end": pd.Timestamp("2021-12-01"),
        "color": "tab:red",
    },
]

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    applications_with_indicators["date"],
    applications_with_indicators["applications"],
    color="tab:blue",
    linewidth=1.5,
    label="First-time applications",
)

for period in indicator_plot_periods:
    ax.axvspan(
        period["start"],
        period["end"] + pd.offsets.MonthEnd(0),
        color=period["color"],
        alpha=0.18,
        label=period["label"],
    )

ax.set_title(
    "Italy first-time asylum applications with descriptive event-period indicators"
)
ax.set_xlabel("Month")
ax.set_ylabel("First-time applications")
ax.legend(loc="upper left")
ax.grid(alpha=0.25)

plt.tight_layout()


The indicators do not attempt to explain every movement in the series. They mark only three broad periods that are historically relevant and defined before model fitting.

We do not add a dedicated post-2022 indicator, even though the held-out test period contains a high-application phase. Adding such an indicator would risk building the test-period pattern into the model by hand. The SARIMA and SARIMAX models should instead be evaluated on whether they can forecast the held-out period without being given a special post-2022 marker.

The next section fits the SARIMAX model using these indicators as exogenous variables.


## 11. Fit and evaluate SARIMAX

The previous section defined three descriptive event-period indicators. We now use those indicators in a SARIMAX model and test whether adding them improves forecast performance.

The comparison is straightforward:

```text
Selected SARIMA
vs.
Selected SARIMA structure + descriptive event-period indicators
```

The selected SARIMA order is:

```text
SARIMA(0, 1, 2)(1, 1, 1, 12)
```

The SARIMAX model uses the same order and seasonal order, but adds the three event-period indicators as exogenous variables. We do not run another parameter search. The purpose is only to test whether accounting for the three defined event periods improves held-out forecast performance.


### 11.1 Prepare exogenous variables

The SARIMAX model uses the three event-period indicators defined above. They are split into training and test-period exogenous variables using the same train/test dates as the forecasting evaluation.


In [ ]:
indicator_columns = [
    "european_refugee_crisis",
    "central_med_route_shift",
    "covid_disruption",
]

train_with_indicators = applications_with_indicators.loc[
    applications_with_indicators["date"].between(train_start, train_end)
].copy()

test_with_indicators = applications_with_indicators.loc[
    applications_with_indicators["date"].between(test_start, test_end)
].copy()

exog_train = train_with_indicators.set_index("date")[indicator_columns].asfreq("MS")
exog_test = test_with_indicators.set_index("date")[indicator_columns].asfreq("MS")

assert len(exog_train) == len(training_series)
assert len(exog_test) == len(test)
assert exog_train.index.equals(training_series.index)
assert exog_test.index.equals(test.set_index("date").index)
assert (exog_test.sum() == 0).all()


The test period is not given a special post-2022 marker. This keeps the comparison disciplined: SARIMAX can use the three historical event-period indicators during fitting, but it is not told directly that the held-out period is a high-application phase.


### 11.2 Fit the SARIMAX model

We fit one SARIMAX model on the training period only. It uses the selected SARIMA structure and the three descriptive event-period indicators as exogenous variables.


In [ ]:
sarimax_model = SARIMAX(
    endog=training_series,
    order=selected_sarima_order,
    seasonal_order=selected_sarima_seasonal_order,
    exog=exog_train,
    enforce_stationarity=False,
    enforce_invertibility=False,
)

sarimax_results = sarimax_model.fit(disp=False)


In [ ]:
sarimax_model_info = pd.DataFrame(
    [
        {
            "Model": "SARIMAX",
            "Non-seasonal order": str(selected_sarima_order),
            "Seasonal order": str(selected_sarima_seasonal_order),
            "Exogenous variables": ", ".join(indicator_columns),
            "Training period": (
                f"{train['date'].min():%Y-%m} "
                f"to {train['date'].max():%Y-%m}"
            ),
            "Test period": (
                f"{test['date'].min():%Y-%m} "
                f"to {test['date'].max():%Y-%m}"
            ),
            "AIC": sarimax_results.aic,
            "BIC": sarimax_results.bic,
        }
    ]
).assign(
    AIC=lambda df: df["AIC"].round(1),
    BIC=lambda df: df["BIC"].round(1),
)

sarimax_model_info


The SARIMAX model has an AIC of **2580.4** and a BIC of **2604.7**. Both are higher than the selected SARIMA model, whose AIC was **2579.2** and BIC was **2594.3**. This means the additional event-period indicators do not improve the penalised in-sample fit enough to justify the extra complexity under these criteria.

AIC and BIC describe in-sample model fit with a complexity penalty. They are useful, but they are not the main basis for choosing the model in this notebook. The main comparison remains held-out forecast performance on the test period.


### 11.3 Forecast the held-out test period

We now forecast the same held-out period used earlier: March 2023 to February 2026.


In [ ]:
sarimax_prediction = sarimax_results.get_forecast(
    steps=len(test),
    exog=exog_test,
)

sarimax_prediction_interval = sarimax_prediction.conf_int(alpha=0.20)

sarimax_forecast = pd.DataFrame(
    {
        "date": test["date"].to_numpy(),
        "actual_applications": test["applications"].to_numpy(),
        "sarimax_forecast": sarimax_prediction.predicted_mean.to_numpy(),
        "sarimax_lower_80": sarimax_prediction_interval.iloc[:, 0].to_numpy(),
        "sarimax_upper_80": sarimax_prediction_interval.iloc[:, 1].to_numpy(),
    }
)


In [ ]:
plot_forecast_with_interval(
    sarimax_forecast,
    forecast_column="sarimax_forecast",
    lower_column="sarimax_lower_80",
    upper_column="sarimax_upper_80",
    title="SARIMAX forecast over the held-out test period",
    forecast_label="SARIMAX forecast",
)


The SARIMAX forecast follows the held-out test period at roughly the same broad level as the selected SARIMA forecast. It remains smoother than the observed monthly series and does not materially improve the fit to sharper month-to-month movements, including the pronounced dip in 2025.

The interval is again model-based and should be read in the same limited sense as the earlier SARIMA intervals.


### 11.4 Compare SARIMAX with earlier models

The final step is to compare SARIMAX forecast accuracy with the earlier baselines and SARIMA models.


In [ ]:
sarimax_metric = evaluate_forecast(
    sarimax_forecast["actual_applications"],
    sarimax_forecast["sarimax_forecast"],
    "SARIMAX",
)

sarimax_model_comparison_metrics = pd.concat(
    [
        selected_model_comparison_metrics,
        pd.DataFrame([sarimax_metric]),
    ],
    ignore_index=True,
)

display_metric_table(sarimax_model_comparison_metrics)


The SARIMAX model has an MAE of **1532.1** and an RMSE of **2023.0** over the held-out test period. This is slightly worse than the selected SARIMA model, which had an MAE of **1526.4** and an RMSE of **2020.6**.

The same conclusion appears in the model-fit criteria. SARIMAX has a higher AIC and BIC than the selected SARIMA model. The additional event-period indicators therefore do not improve the model enough to justify the extra complexity under these criteria.

This does not mean the event-period indicators were meaningless. They were a transparent way to test whether three historically relevant periods helped the model estimate the training-period structure. In this setup, however, they did not translate into better held-out forecast performance.

The result is a useful modelling lesson: historically plausible indicators do not automatically improve forecasts. A variable can make substantive sense and still fail to add predictive value once it is tested against held-out data.

The comparison therefore supports carrying forward the selected SARIMA model, not the SARIMAX model, to the future projection section. The decision is based mainly on held-out MAE, with RMSE and the information criteria as secondary checks.



# Part IV - Projection and interpretation


## 12. Future projection

We now refit the selected SARIMA model on the full observed series and project 12 months ahead.

The selected structure is:

```text
SARIMA(0, 1, 2)(1, 1, 1, 12)
```

The projection covers March 2026 to February 2027.


In [ ]:
full_series = applications.set_index("date")["applications"].asfreq("MS")

final_sarima_model = SARIMAX(
    endog=full_series,
    order=selected_sarima_order,
    seasonal_order=selected_sarima_seasonal_order,
    enforce_stationarity=False,
    enforce_invertibility=False,
)

final_sarima_results = final_sarima_model.fit(disp=False)

future_prediction = final_sarima_results.get_forecast(steps=12)
future_prediction_interval = future_prediction.conf_int(alpha=0.20)

future_projection = pd.DataFrame(
    {
        "date": pd.date_range(
            start=applications["date"].max() + pd.DateOffset(months=1),
            periods=12,
            freq="MS",
        ),
        "forecast": future_prediction.predicted_mean.to_numpy(),
        "lower_80": future_prediction_interval.iloc[:, 0].to_numpy(),
        "upper_80": future_prediction_interval.iloc[:, 1].to_numpy(),
    }
)

assert len(future_projection) == 12
assert future_projection["date"].iloc[0] == applications["date"].max() + pd.DateOffset(months=1)
assert future_projection["date"].iloc[-1] == applications["date"].max() + pd.DateOffset(months=12)


In [ ]:
future_projection_table = future_projection.assign(
    Month=future_projection["date"].dt.strftime("%Y-%m"),
    **{
        "Projected applications": future_projection["forecast"],
        "Lower 80%": future_projection["lower_80"],
        "Upper 80%": future_projection["upper_80"],
    },
)[["Month", "Projected applications", "Lower 80%", "Upper 80%"]]

total_row = pd.DataFrame(
    [
        {
            "Month": "Total",
            "Projected applications": future_projection["forecast"].sum(),
            "Lower 80%": future_projection["lower_80"].sum(),
            "Upper 80%": future_projection["upper_80"].sum(),
        }
    ]
)

future_projection_table = pd.concat(
    [future_projection_table, total_row],
    ignore_index=True,
)

for column in ["Projected applications", "Lower 80%", "Upper 80%"]:
    future_projection_table[column] = (
        future_projection_table[column].round(0).astype(int).map("{:,}".format)
    )

future_projection_table


The lower and upper values are monthly pointwise model-based interval bounds. The total row sums the monthly columns as a compact reference; it should not be read as a formal probability interval for the 12-month total.


In [ ]:
annual_observed_for_projection = (
    applications.assign(year=applications["date"].dt.year)
    .groupby("year", as_index=False)["applications"]
    .sum()
    .rename(columns={"applications": "observed_applications"})
)

projected_2026_total = future_projection.loc[
    future_projection["date"].dt.year == 2026,
    "forecast",
].sum()

annual_observed_for_projection["projected_applications"] = 0.0
annual_observed_for_projection.loc[
    annual_observed_for_projection["year"] == 2026,
    "projected_applications",
] = projected_2026_total

fig, ax = plt.subplots(figsize=(10, 4))

ax.bar(
    annual_observed_for_projection["year"].astype(str),
    annual_observed_for_projection["observed_applications"],
    label="Observed applications",
)

ax.bar(
    annual_observed_for_projection["year"].astype(str),
    annual_observed_for_projection["projected_applications"],
    bottom=annual_observed_for_projection["observed_applications"],
    label="Projected applications, Mar-Dec 2026",
    color="tab:orange",
)

ax.set_title("Annual observed applications with projected 2026 continuation")
ax.set_xlabel("Year")
ax.set_ylabel("Applications")
ax.tick_params(axis="x", rotation=45)
ax.legend()
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
plt.show()


In [ ]:
observed_projection_window = applications.loc[
    applications["date"] >= pd.Timestamp("2018-01-01")
]
forecast_start = future_projection["date"].iloc[0]

fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(
    observed_projection_window["date"],
    observed_projection_window["applications"],
    label="Observed applications",
)
ax.plot(
    future_projection["date"],
    future_projection["forecast"],
    label="12-month SARIMA projection",
)
ax.fill_between(
    future_projection["date"],
    future_projection["lower_80"].to_numpy(),
    future_projection["upper_80"].to_numpy(),
    alpha=0.2,
    label="80% model-based interval",
)
ax.axvline(
    forecast_start,
    color="black",
    linestyle="--",
    linewidth=1,
    label="Forecast start",
)

ax.set_title("Selected SARIMA 12-month projection")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.legend()
ax.grid(alpha=0.25)
plt.show()


The 12-month projection remains close to the recent high level of the observed series. The point forecast fluctuates month to month, reflecting the seasonal structure of the model, but it does not suggest a return to the lower application levels seen earlier in the series.

Across March 2026 to February 2027, the projected total is about **130,690 first-time applications**. The summed monthly lower and upper 80% bounds are approximately **94,818** and **166,562**, respectively. These totals should be read as compact references based on the monthly intervals, not as a formal probability interval for the annual total.

The model-based interval is wide and generally widens over the projection horizon. This reflects increasing uncertainty as the forecast moves further away from the last observed month.